# 03 · groupby / merge / pivot_table：把数据从行变成表

**时间预算**：3 小时。前 1 小时 40 分钟跟着跑，剩下做练习。

前两课我们一直在"看"数据——看形状、看缺失、看分布。**但看是手段，结论才是目的。**

而结论几乎总是**表状的**：

> 1 等舱的生存率是 63%，3 等舱是 24%

这句话是一张 3 行的表。可数据是 891 行的。**从 891 行变成 3 行，靠的就是今天这三个工具。**

上次你做练习 5 时被"基准"打翻——那件事的正解就在这里：**`groupby` 就是算基准的工具**。

学完你应该能回答：

1. 分组聚合的三步是什么？为什么结果是一张"新表"？
2. `merge` 之后行数变多了，是 bug 还是正常？
3. `pivot_table` 和 `groupby` 到底差在哪？

## 1. 为什么需要 groupby

先看**没有它**的时候有多笨。上次练习 3 你就是这么写的：

In [ ]:
import pandas as pd

pd.set_option("display.max_columns", 60)
df = pd.read_csv("../data/raw/titanic.csv")

# 笨办法：有几组就写几遍
for pclass in [1, 2, 3]:
    subset = df[df["Pclass"] == pclass]
    print(f"{pclass} 等舱: {len(subset):>3} 人, 生存率 {subset['Survived'].mean():.3f}")

三组还能忍。**但如果要按性别 × 舱位 × 登船港分（3×2×3 = 18 组）呢？**
写 18 遍 `for` 循环？再加一个维度就爆炸了。

`groupby` 把这件事变成一句话。它内部做的是三步，这个模型值得记住：

```
split（按 key 把 891 行切成几堆）
   ↓
apply（每堆各自算一个数）
   ↓
combine（把结果拼成一张新表）
```

> 这个词组（split-apply-combine）是数据分析的通用套路，
> 后面学 SQL 的 `GROUP BY`、PySpark 的 `groupBy`、甚至 map-reduce，都是同一个东西。

In [ ]:
# 一句话版本
df.groupby("Pclass")["Survived"].mean()

读法是：**按 Pclass 分组，取出 Survived 这一列，求平均**。

结果是一张**新表**（严格说是 Series），索引变成了 `Pclass` 的取值。
这个细节很重要：**你原来的 891 行不见了，现在只剩 3 行**——每行代表一个组。

In [ ]:
result = df.groupby("Pclass")["Survived"].mean()
print(type(result).__name__)
print("索引:", result.index.tolist(), "  <- 不是 0,1,2，而是分组键的取值")
print("长度:", len(result), "  <- 从 891 行压成了 3 行")
print()
print(result)

### 多个分组键：结果变成多级索引

传一个列表进去就能按多列分组。这才是 `groupby` 真正超过手写循环的地方。

In [ ]:
by_two = df.groupby(["Pclass", "Sex"])["Survived"].mean()
print(by_two)
print()
print("索引类型:", type(by_two.index).__name__, " <- 多级索引，两层")
print("总共", len(by_two), "组")

多级索引看着别扭，用 `reset_index()` 把它压回普通表格就好——**这一步做分析时几乎每次都要做**。

In [ ]:
flat = by_two.reset_index()
flat.columns = ["舱位", "性别", "生存率"]
print(flat.round(3).to_string(index=False))
print()
print("  ^ 这就是一张能直接放进报告的结论表了")

### 一次算多个统计量：agg

实际分析里你很少只要平均值——**通常同时要看人数和比例**，因为"3 等舱生存率 24%"
和"3 等舱 491 人里活了 119 人"是两种信息量。

In [ ]:
# 写法 1：传一个统计函数列表（结果列名是自动生成的）
df.groupby("Pclass")["Survived"].agg(["count", "sum", "mean"]).round(3)

In [ ]:
# 写法 2：命名聚合（推荐）—— 直接给结果列起名字，可读性天差地别
df.groupby("Pclass").agg(
    人数=("PassengerId", "count"),
    存活人数=("Survived", "sum"),
    生存率=("Survived", "mean"),
    平均票价=("Fare", "mean"),
    年龄中位数=("Age", "median"),
).round(3)

**命名聚合这种写法要养成习惯。** 它的语法是 `新列名=(原列名, 统计函数)`。
好处是——你三个月后回来看这张表，不用猜 `Survived_mean` 到底是什么。

看一下这张表能读出什么：1 等舱平均票价 84，3 等舱 13.7，差了 6 倍；
而 1 等舱年龄中位数 37 岁，3 等舱 24 岁。**舱位不只代表价钱，还代表年龄结构。**
这就是为什么后面做模型时，`Pclass` 是个强特征。

## 2. merge：把两张表拼起来

现实里的数据**很少完整地躺在同一张表里**。通常是：

- 一张事实表（几百行记录，比如每个乘客的行程）
- 几张维度表（几十行的说明，比如港口代码对应哪个城市）

你要把它们拼起来才能分析。先造一张小的维度表：

In [ ]:
ports = pd.DataFrame(
    {
        "Embarked": ["S", "C", "Q"],
        "港口": ["Southampton", "Cherbourg", "Queenstown"],
        "国家": ["英国", "法国", "爱尔兰"],
    }
)
print(ports.to_string(index=False))
print()
print("Titanic 里的 Embarked 只有代码，看不出是哪个国家的港口 —— 所以要连表")

In [ ]:
merged = df.merge(ports, on="Embarked", how="left")

print("连表前的行数:", len(df))
print("连表后的行数:", len(merged))
print("列数:", df.shape[1], "->", merged.shape[1])
print()
print(merged[["Name", "Embarked", "港口", "国家"]].head(4).to_string(index=False))

### 连表之后第一件事：对比行数

上面行数没变（891 → 891），这是我们期望的：`ports` 表里每个 `Embarked` 只出现一次，
所以每个乘客刚好匹配到一条。这种关系叫**多对一**（多个事实行 对 一个维度行）。

**但如果维度表本身有重复值呢？** 看下面——这是数据工作里最常见的一种翻车。

In [ ]:
# 假设录入时不小心把 S 录了两遍（真实工作中非常常见）
ports_bad = pd.DataFrame(
    {
        "Embarked": ["S", "S", "C", "Q"],
        "国家": ["英国", "英国", "法国", "爱尔兰"],
    }
)

bad_merge = df.merge(ports_bad, on="Embarked", how="left")
print("连表前:", len(df), "行")
print(
    "连表后:", len(bad_merge), "行", "  <- 凭空多了", len(bad_merge) - len(df), "行！"
)
print()
print("多出来的原因：S 港的 644 个乘客，每人匹配到了 2 行港口信息")

**这个错误的可怕之处**：它不报错，也不影响你算单行。但接下来
`len(bad_merge)`、`bad_merge["Survived"].mean()`、`bad_merge["Fare"].sum()` **全部会算错**——
因为 S 港的人被重复计算了。你会发现生存率、总票价、人数全都不对，而且找半天找不到原因。

**防线有两个**：

**第一个，也是最简单的：连表前后对比行数。** 行数变了就必须解释清楚为什么。

In [ ]:
def safe_merge(left, right, how="left", **kwargs):
    """连表并自动检查行数——把这两个动作捆在一起，就不会忘。"""
    out = left.merge(right, how=how, **kwargs)
    print(f"行数: {len(left):,} -> {len(out):,}", end="")
    if len(out) != len(left):
        print(f"   ⚠️ 变化了 {len(out) - len(left):+,} 行，必须查清原因")
    else:
        print("   ✅ 没变")
    return out


_ = safe_merge(df, ports, on="Embarked")
_ = safe_merge(df, ports_bad, on="Embarked")

**第二个防线更硬：让 pandas 替你检查关系。** `validate=` 参数会验证表之间的关系，
不满足就直接报错，而不是默默给你一张错表。

In [ ]:
# 正常情况：多对一，通过
_ = df.merge(ports, on="Embarked", how="left", validate="many_to_one")
print("ports 表通过了 many_to_one 检查")
print()

# 有重复值的情况：直接报错，不给你错的结果
try:
    df.merge(ports_bad, on="Embarked", how="left", validate="many_to_one")
except Exception as exc:
    print("ports_bad 表被拦下来了：")
    print(f"  {type(exc).__name__}: {str(exc)[:150]}")

**以后连表就带上 `validate=`。** 这一个参数的代价是几秒钟，收益是避免一次
"数字怎么不对"的深夜排查。你在练习里可以先用 `many_to_one`——大多数维度表都该是它。

### 四种连接方式

| `how=` | 保留谁 | 什么时候用 |
| :--- | :--- | :--- |
| `inner` | 只保留两边都匹配上的 | 只要交集（会丢行） |
| `left` | 保留左表全部 | **最常用**：以事实表为准，维度信息有就带上 |
| `right` | 保留右表全部 | 少见 |
| `outer` | 两边都保留 | 查"哪边缺了"的时候 |

**新手最容易踩的坑**：以为 `left` 连接不会丢数据。它确实不丢**左表**的行，
但如果匹配不上，右边那些列会是 `NaN`——**看起来有值其实是空**。
连完记得检查一下新列有没有 NaN：

In [ ]:
# indicator=True 会加一列告诉你每行是匹配上的还是没匹配上
checked = df.merge(ports, on="Embarked", how="left", indicator=True)
print(checked["_merge"].value_counts().to_string())
print()
print("  left_only 就是没匹配上的行 —— 那 2 行正是 Embarked 缺失的乘客")

## 3. pivot_table：把行摊成矩阵

`groupby` 出来的表是"长"的——每个组合一行：

```
Pclass  Sex      Survived
1       female   0.968
1       male     0.369
2       female   0.921
...
```

**人眼不擅长读长表。** 我们擅长读矩阵——行是舱位、列是性别，一眼就能比较。
`pivot_table` 就是干这个的。

In [ ]:
matrix = pd.pivot_table(
    df,
    values="Survived",
    index="Pclass",  # 变成行
    columns="Sex",  # 变成列
    aggfunc="mean",
)
matrix.round(3)

这张表比刚才的长表好用得多：**横着看是性别差异，竖着看是舱位差异，交叉处是组合效应。**

它和你练习 1 里想找的东西是同一个："女性生存率明显高于男性，而且头等舱更明显"——
这句话只有在这种矩阵里才看得清。

In [ ]:
# 加上总计（margins）和空组合填充
rich = pd.pivot_table(
    df,
    values=["Survived", "Fare"],
    index="Pclass",
    columns="Embarked",
    aggfunc={"Survived": "mean", "Fare": "mean"},
    margins=True,
    margins_name="全船",
)
rich.round(2)

### pivot_table 和 groupby 是什么关系

**它们是同一件事的两种摆法。** 证明给你看：

In [ ]:
# 路线 A：groupby + unstack
route_a = df.groupby(["Pclass", "Sex"])["Survived"].mean().unstack()

# 路线 B：pivot_table
route_b = pd.pivot_table(
    df, values="Survived", index="Pclass", columns="Sex", aggfunc="mean"
)

print("两条路线结果相同吗:", route_a.equals(route_b))
print()
print(route_a.round(3))

既然等价，怎么选？

- **想先算出一堆指标、再决定怎么摆** → 用 `groupby`，然后 `reset_index()` 或 `unstack()`
- **上来就知道要一张矩阵** → 用 `pivot_table`，写起来更直接

### 三个工具的关系

| 工具 | 干什么 | 结果形状 | 一句话记忆 |
| :--- | :--- | :--- | :--- |
| `groupby` | 按组算汇总 | **长表**（每组一行） | 把 891 行压成 3 行 |
| `pivot_table` | 把某列的值摊成列 | **宽表**（矩阵） | 把某种长表摆成矩阵 |
| `merge` | 横向拼两张表 | **更宽的表** | 加列，不加行（正常情况下） |

前两个是**纵向压缩**（减少行），第三个是**横向扩展**（增加列）。这就是数据分析的两条腿。

## 4. 实战：连续变量先分箱，再分组

`groupby` 只能按**离散的取值**分组。年龄是连续的（0.42 到 80），怎么办？

答案是**先分箱**（binning），把它切成几段。这是特征工程里的常规操作，
因为"每差一岁的生存率"没有意义，但"每 10 岁一段"就有意义了。

In [ ]:
bins = [0, 10, 20, 40, 60, 100]
labels = ["0-10岁", "10-20岁", "20-40岁", "40-60岁", "60岁以上"]

df["年龄段"] = pd.cut(df["Age"], bins=bins, labels=labels)

counts = df["年龄段"].value_counts().sort_index()
print(counts.to_string())
print()
print("五类加起来:", counts.sum(), "  而总行数是:", len(df))
print(
    f"少了 {len(df) - counts.sum()} 人 —— 正是 Age 缺失的那些（value_counts 默认不统计 NaN）"
)

In [ ]:
df.groupby("年龄段", observed=True).agg(
    人数=("PassengerId", "count"),
    生存率=("Survived", "mean"),
).round(3)

看到那个经典结论了吗：**小孩生存率明显高于成人**（"妇女儿童优先"）。
这个结论你在练习里是用筛选做的，现在用分组做——**数据量再大十倍也是一句话。**

另外注意 `observed=True`：分箱会产生"类别类型"的列，那些一次都没出现的箱子
默认也会占一行（人数 0）。加上这个参数就只显示真实存在的组。

## 5. 练习（约 1 小时）

老规矩：每格都要有输出，第 4、5 题要写文字结论。

### 练习 1（热身）

按登船港（`Embarked`）分组，一次算出：**人数、平均票价、生存率**（用命名聚合）。

哪個港口的生存率最高？

In [ ]:
# 你的代码
a = df.groupby("Embarked").agg(
    人数=("PassengerId", "count"),
    平均票价=("Fare", "mean"),
    生存率=("Survived", "mean"),
)
print("生存率最高的港口")
print(f"{a.sort_values('生存率', ascending=False).head(1)}")

### 练习 2（两种摆法）

用 `groupby` + `unstack` 和 `pivot_table` 两种写法，得到同一张"**舱位 × 性别**的生存率矩阵"。

然后用 `.equals()` 验证两者是否完全相同。

In [ ]:
# 你的代码
a = df.groupby(["Pclass", "Sex"])["Survived"].mean().unstack()
print(a.round(3))

b = pd.pivot_table(df, values="Survived", index="Pclass", columns="Sex")
print(b.round(3))
print(a.equals(b))

### 练习 3（分箱 + 分组）

把票价（`Fare`）分成 4 段：`[0, 10, 30, 100, 600]`，标签叫 `["低价", "中价", "高价", "天价"]`。

然后算出每个价位的**人数和生存率**。

In [ ]:
# 你的代码
bins = [0, 10, 30, 100, 600]
labels = ["低价", "中价", "高价", "天价"]
print(len(df))
df["价格"] = pd.cut(df["Fare"], bins=bins, labels=labels, include_lowest=True)
print(df["价格"].value_counts(dropna=False))

a = df.groupby("价格").agg(人数=("PassengerId", "count"), 生存率=("Survived", "mean"))
print(a.round(3))
# 注意：票价最小是 0，bins 的第一个边界要能包住它

### 练习 4（merge 的关系，要写结论）

下面这张表记录了每个家庭的规模对应的说明。把它和 `df` 连起来。

**要求**：① 连表前后对比行数；② 用 `validate=` 检查关系；③ 写下你觉得该用哪种 `how=`，为什么。

In [ ]:
family_size_note = pd.DataFrame(
    {
        "FamilySize": [1, 2, 3, 4, 5, 6, 7, 8, 11],
        "类型": [
            "独自一人",
            "夫妻/亲子",
            "小家庭",
            "小家庭",
            "大家庭",
            "大家庭",
            "大家庭",
            "大家庭",
            "超大家庭",
        ],
    }
)

# 先算每个人的家庭规模
df["FamilySize"] = df["SibSp"] + df["Parch"] + 1

# 你的代码：连表 + 对比行数 + validate 检查
a = df.merge(family_size_note, on="FamilySize", how="left", validate="many_to_one")
print(f"连表前有{len(df)}行，连表后有{len(a)}行")
# 我的结论（how 用哪个、为什么）：以df为事实表，family_size_note表只是维度信息

### 练习 5（找出会膨胀的连表，要写结论）

下面这张表**故意有重复**。先预测连表后行数会变成多少，再实际跑一遍验证。

然后写下：**为什么`validate="many_to_one"` 会拦住它？如果不用 validate，
这个错误会在后面哪一步暴露出来？**

In [ ]:
cabin_deck = pd.DataFrame(
    {
        "Cabin": ["C85", "C85", "B42"],
        "甲板": ["C 甲板", "C 甲板（重复录入）", "B 甲板"],
    }
)

# 先猜：连表后行数会是多少？
# 我的猜测：连表后会多出（df表中Cabin值C85的个数）条数据，因为后表中出现了两次C85，故前表中值为C85的元素会匹配两次后表

# 你的代码：实际连一次，对比行数，再试 validate
a = df.merge(cabin_deck, on="Cabin", how="left")
print(len(a))
try:
    b = df.merge(cabin_deck, on="Cabin", how="left", validate="many_to_one")
except Exception as exc:
    print(f"{type(exc).__name__}:{str(exc)[:150]}")

# 我的结论：validate = "many_to_one"表示了前表和后表的对应关系为对多一，当发现前表中对应后表的两个值时就会被拦住，如果不用validate也不会报错，但会给你返回一个数据已经出错了的表，会导致后面得到的数据都不准确，当查看表的长度的时候就会发现表变长了，这也暴露了问题所在

### 练习 6（进阶：把今天的三个工具串起来）

做出一张**"性别 × 年龄段"的生存率矩阵**（行是性别，列是年龄段，值是生存率）。

要求用上 `pd.cut`、`pivot_table`，最后加上 `margins=True` 的总计行。

In [ ]:
# 你的代码
bins = [0, 10, 20, 40, 60, 100]
labels = ["0-10", "10-20", "20-40", "40-60", "60以上"]
df["年龄段"] = pd.cut(df["Age"], bins=bins, labels=labels)

b = pd.pivot_table(
    df,
    values="Survived",
    index="Sex",
    columns="年龄段",
    aggfunc="mean",
)
b

## 6. 总结卡片

1. **`groupby` 的三步**：split → apply → combine。结果是一张新表，索引是分组键。
2. **命名聚合**：`agg(新名=(旧列, 函数))`。让人看得懂的表，才是好表。
3. **连表后立刻对比行数**，并带上 `validate=`。行数变了必须有解释，否则后面所有统计都是错的。
4. **长表和矩阵**：`groupby` 给长表，`pivot_table` 给矩阵。人眼读矩阵，程序读长表。
5. **连续变量先分箱**再分组（`pd.cut`）。

---

### 下一课预告

今天你学会把数据**算成结论**。但结论要给人看，就得画出来。

下一课是 `matplotlib`：**一张图顶一千行代码**，但也有一千种画错的方式。

带着一个问题来：为什么同一份数据，换一种图的画法，能得出完全相反的结论？

## 7. 卡住时的提示（先别看）

**练习 1**：命名聚合的写法是 `agg(列名=("原列", "函数"))`。生存率是对 `Survived` 求 mean。
别忘了 `Embarked` 有 2 个缺失值——想想它们会变成单独一组还是被丢掉。

**练习 2**：`groupby(["A","B"])["C"].mean()` 之后再 `.unstack()` 就会得到矩阵。
`pivot_table` 那边 index 和 columns 要分别对应 A 和 B。最后用 `.equals()` 比。

**练习 3**：`pd.cut(列, bins=[...], labels=[...])`。注意 bins 要覆盖所有值——
票价最小值是 0，最大值 512，所以最后一个边界要大于 512。

**练习 4**：`how` 该用哪个——问自己"我要保留谁的全部信息"。
`validate` 的参数是 `"many_to_one"`（多个乘客 对 一个家庭规模）。
另外注意：如果有人 FamilySize 是 9 或 10，表里没有，连完会是 NaN——这就是 `left` 的代价。

**练习 5**：`cabin_deck` 里 C85 出现了两次。连表时 Titanic 里每个 C85 客舱的人会匹配上 2 行。
先想清楚"哪些人会受影响、多出来几行"再跑。
至于"会在哪一步暴露"——想想你连完表之后最可能做的第一个统计是什么。

**练习 6**：先在 df 上做两件事：`pd.cut` 造年龄段（第 4 节已经造过 `年龄段` 列，可以直接用），
然后 `pivot_table(index="Sex", columns="年龄段", values="Survived", aggfunc="mean", margins=True)`。
如果报错说需要 `observed`，加上它。